# Analysis - the state of a molecule, coloured by its odor

Issue #11. 115 validation molecules that carry exactly one of eight frequent odor tokens. The mean of the last-layer
state over the molecule tokens, in three settings: the generator asked for the molecule's own tokens, the generator
asked for nothing, and the frozen pretrained decoder.

Three numbers per setting, because the picture is not the evidence: a silhouette in 256 dimensions, a
leave-one-out k-nearest-neighbour vote, and the share the most common token would get by guessing.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data = json.loads((ROOT / "output/analysis/hidden_states.json").read_text())
tokens = data["tokens"]
labels = np.array(data["labels"])
print(data["molecules"], "molecules,", len(tokens), "tokens, chance", round(data["chance"], 3))
print(pd.Series(data["per_token"]).to_string())
pd.DataFrame({"silhouette": data["silhouette"], "kNN": data["knn_accuracy"]}).round(3)


## The three settings on one projection

PCA fitted on the frozen decoder's states, so all three panels share the axes and the panels can be compared. Colour
is the one frequent odor token the molecule carries.

In [ ]:
figure = plt.figure(figsize=(15, 9))
grid = figure.add_gridspec(2, 3, height_ratios=(1.05, 0.95))
panels = [figure.add_subplot(grid[0, column]) for column in range(3)]
colours = plt.get_cmap("tab10")
points = {name: np.array(data["coordinates"][name]) for name in ("own", "empty", "frozen")}
low = np.min([values.min(axis=0) for values in points.values()], axis=0)
high = np.max([values.max(axis=0) for values in points.values()], axis=0)
for ax, name, title in zip(panels, ("own", "empty", "frozen"), ("own request", "empty request", "pretrained decoder"), strict=True):
    for index, token in enumerate(tokens):
        chosen = labels == index
        ax.scatter(points[name][chosen, 0], points[name][chosen, 1], s=16, color=colours(index), label=token)
    ax.set_title(f"{title}\nkNN {data['knn_accuracy'][name]:.2f}, silhouette {data['silhouette'][name]:+.3f}")
    ax.set_xlim(low[0] - 0.05, high[0] + 0.05)
    ax.set_ylim(low[1] - 0.05, high[1] + 0.05)
    ax.set_xticks([])
    ax.set_yticks([])
panels[0].legend(frameon=False, fontsize=7, markerscale=1.5)
movement = figure.add_subplot(grid[1, :2])
for index, token in enumerate(tokens):
    chosen = np.where(labels == index)[0]
    for row in chosen:
        movement.annotate("", xy=points["own"][row], xytext=points["empty"][row], arrowprops={"arrowstyle": "->", "color": colours(index), "alpha": 0.55, "lw": 0.8})
    start = points["empty"][chosen].mean(axis=0)
    end = points["own"][chosen].mean(axis=0)
    movement.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "-|>", "color": colours(index), "lw": 2.0})
movement.set_xlim(low[0] - 0.05, high[0] + 0.05)
movement.set_ylim(low[1] - 0.05, high[1] + 0.05)
movement.set_title(f"What the request moves: empty to own, median {data['displacement']:.2f} in 256 dimensions")
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", linestyle="", color=colours(index), label=token) for index, token in enumerate(tokens)]
movement.legend(handles=handles, loc="upper right", fontsize=8, frameon=False, ncols=2)
movement.set_xticks([])
movement.set_yticks([])
votes = figure.add_subplot(grid[1, 2])
confusion = np.array(data["confusion"])
votes.imshow(confusion, cmap="Blues")
for row in range(confusion.shape[0]):
    for column in range(confusion.shape[1]):
        if confusion[row, column]:
            votes.text(column, row, str(confusion[row, column]), ha="center", va="center", fontsize=7)
votes.set_xticks(range(len(tokens)), tokens, rotation=90, fontsize=7)
votes.set_yticks(range(len(tokens)), tokens, fontsize=7)
votes.set_xlabel("nearest neighbour vote")
votes.set_title("Own request: where the votes go")
plt.tight_layout()
plt.show()


## Reading.

The mean state of a molecule does not form eight clusters. The silhouette is -0.005 with the molecule's own request,
-0.049 with an empty request and -0.052 with the frozen decoder: at zero, or slightly below, a molecule is as close
to its own colour as to another.

The local neighbourhood tells a different story. A vote among the five nearest molecules puts the right token 57
percent of the time with the own request, 38 percent with an empty one and 38 percent with the frozen decoder,
against 22 percent for always answering with the most common token. So the token is there in the fine structure and
absent in the coarse one, and the request adds 19 points to the vote without creating clusters.

The same thing is visible in the movement panel. Every molecule moves when it is asked for, by a median of 1.39 in
256 dimensions, and the eight mean arrows point in different directions. The movement is real and it does not
gather the molecules into groups.

## Verdict

The conditioning is a direction in a crowded space, not a partition of it. That fits the adapter analysis: the
request adds 10 to 17 percent of the state, enough to change which molecule token comes next, not enough to move a
molecule into a cluster of its own.